
# 03 - Enfoque del Problema Predictivo

## Evaluación 2 - Definición del objetivo de modelado

Este notebook tiene como propósito explicar **por qué se plantea un modelo predictivo**, qué variable podría utilizarse como objetivo y cómo se conecta el dataset final generado en Kedro con la etapa de Machine Learning.

En esta etapa **todavía no se entrenan modelos**.  
El objetivo es entender primero qué se quiere predecir y por qué.


In [1]:

%load_ext kedro.ipython

import pandas as pd
import numpy as np


The kedro.ipython extension is already loaded. To reload it, use:
  %reload_ext kedro.ipython



## 1. Contexto del problema

El proyecto trabaja con datos de una operación logística compuesta por envíos, rutas, vehículos e incidencias.

Desde una perspectiva de negocio, una empresa logística necesita anticipar problemas operacionales antes de que ocurran. Por ejemplo:

- Identificar si un envío podría llegar tarde.
- Estimar cuántos días podría tardar una entrega.
- Detectar rutas o vehículos con mayor riesgo operativo.
- Apoyar la toma de decisiones en planificación, asignación de recursos y prevención de incidencias.

Por eso, después de limpiar e integrar los datos con Kedro, se puede avanzar hacia modelos predictivos.



## 2. Carga del dataset final

Se utiliza el dataset final generado por Kedro:

`dataset_modelo`

Este archivo corresponde a la salida final del flujo:

`datos crudos → limpieza → transformación → validación → dataset para modelado`

Por lo tanto, es el dataset adecuado para comenzar el análisis predictivo.


In [2]:

df = catalog.load("dataset_modelo")

df.head()


[05/07/26 16:05:19] INFO     Loading data from dataset_modelo (CSVDataset)...                  ]8;id=6996215;file://C:\Users\maico\Desktop\ProyectoProgramacionEva\Evaluacion1Programacion\venv\lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=6996216;file://C:\Users\maico\Desktop\ProyectoProgramacionEva\Evaluacion1Programacion\venv\lib\site-packages\kedro\io\data_catalog.py#1053\1053]8;;\

,id_envio,fecha_envio,id_ruta,id_vehiculo,peso_kg,volumen_m3,tipo_carga,estado,fecha_entrega,origen,...,km_recorridos,cantidad_incidencias,costo_total_incidencias,tipo_incidencia_principal,tiene_incidencia,dias_entrega,velocidad_promedio_km_h,uso_capacidad_kg,uso_capacidad_m3,entrega_tardia
0,1.0,2023-01-01,39.0,46.0,12675.8,28.48,peligrosa,entregado,2023-01-05,valparaíso,...,17043.0,0.0,0.0,sin incidencia,0.0,4.0,55.047904,1.26758,1.389268,1
1,2.0,NaN,3.0,27.0,13.6,24.93,peligrosa,entregado,2023-01-05,rancagua,...,88004.0,0.0,0.0,sin incidencia,0.0,7.0,36.934783,0.01360,1.128054,1
2,3.0,2023-01-02,73.0,18.0,2536.0,50.55,refrigerada,entregado,NaN,maipú,...,243705.0,0.0,0.0,sin incidencia,0.0,7.0,90.076923,0.12680,1.299486,1
3,4.0,2023-01-03,49.0,45.0,466.0,13.88,peligrosa,en tránsito,2023-01-07,las condes,...,224573.5,1.0,1747372.0,desvío de ruta,1.0,4.0,86.899471,0.46600,0.548617,1
4,5.0,NaN,17.0,36.0,4928.9,0.44,refrigerada,retrasado,2023-01-07,maipú,...,138259.0,0.0,0.0,sin incidencia,0.0,7.0,86.960894,0.98578,0.008961,1



## 3. Revisión de columnas disponibles

Antes de definir un modelo, es necesario revisar qué variables existen en el dataset.


In [3]:

df.columns.tolist()



[
    'id_envio',
    'fecha_envio',
    'id_ruta',
    'id_vehiculo',
    'peso_kg',
    'volumen_m3',
    'tipo_carga',
    'estado',
    'fecha_entrega',
    'origen',
    'destino',
    'distancia_km',
    'tiempo_estimado_hrs',
    'tipo_via',
    'peaje_total',
    'placa',
    'tipo',
    'capacidad_kg',
    'capacidad_m3',
    'año_fabricacion',
    'estado_vehiculo',
    'km_recorridos',
    'cantidad_incidencias',
    'costo_total_incidencias',
    'tipo_incidencia_principal',
    'tiene_incidencia',
    'dias_entrega',
    'velocidad_promedio_km_h',
    'uso_capacidad_kg',
    'uso_capacidad_m3',
    'entrega_tardia'
]


## 4. Posibles variables objetivo

En Machine Learning, la **variable objetivo** es aquello que queremos predecir.

En este proyecto existen dos posibles enfoques principales:

### 4.1 Clasificación

La clasificación se utiliza cuando queremos predecir una categoría.

En este caso, una variable posible es:

`entrega_tardia`

Esta variable indica si un envío fue tardío o no.

- `0` = entrega normal
- `1` = entrega tardía

Esto permite construir un modelo que responda:

> ¿Este envío tiene riesgo de llegar tarde?

---

### 4.2 Regresión

La regresión se utiliza cuando queremos predecir un valor numérico.

En este caso, una variable posible es:

`dias_entrega`

Esta variable representa la cantidad de días que tardó un envío.

Esto permite construir un modelo que responda:

> ¿Cuántos días podría tardar una entrega?


In [4]:

columnas_interes = ["entrega_tardia", "dias_entrega"]

df[columnas_interes].head()


,entrega_tardia,dias_entrega
0,1,4.0
1,1,7.0
2,1,7.0
3,1,4.0
4,1,7.0



## 5. ¿Por qué `entrega_tardia` necesita un modelo predictivo?

La variable `entrega_tardia` es útil porque representa un problema real de negocio.

Para una empresa logística, saber con anticipación si un envío podría llegar tarde permite:

- Reasignar vehículos.
- Ajustar rutas.
- Informar al cliente.
- Priorizar entregas críticas.
- Reducir costos por incidencias o incumplimientos.

Por eso, `entrega_tardia` es una buena variable objetivo para un modelo de clasificación.


In [5]:

df["entrega_tardia"].value_counts()



entrega_tardia
1    1002
0      28
Name: count, dtype: int64

In [6]:

df["entrega_tardia"].value_counts(normalize=True) * 100



entrega_tardia
1    97.281553
0     2.718447
Name: proportion, dtype: float64


## 6. ¿Por qué `dias_entrega` también puede ser una variable objetivo?

La variable `dias_entrega` permite estimar el tiempo aproximado de entrega.

Esto es útil para planificación logística, porque ayuda a responder:

- ¿Cuánto podría tardar una entrega?
- ¿Qué rutas requieren más tiempo?
- ¿Qué factores influyen en la duración?
- ¿Cómo mejorar la planificación operativa?

Por eso, `dias_entrega` es una buena variable objetivo para un modelo de regresión.


In [7]:

df["dias_entrega"].describe()



count    1030.000000
mean        7.041748
std         0.817213
min         4.000000
25%         7.000000
50%         7.000000
75%         7.000000
max        10.000000
Name: dias_entrega, dtype: float64


## 7. Variables que podrían ayudar a predecir

No todas las columnas sirven como entrada para un modelo.

Algunas variables pueden ayudar a explicar o predecir el resultado, por ejemplo:

- `distancia_km`
- `tiempo_estimado_hrs`
- `peso_kg`
- `volumen_m3`
- `capacidad_kg`
- `capacidad_m3`
- `cantidad_incidencias`
- `tiene_incidencia`
- `uso_capacidad_kg`
- `velocidad_promedio_km_h`

Estas variables representan información operacional del envío, la ruta, el vehículo o las incidencias.


In [8]:

posibles_predictoras = [
    "distancia_km",
    "tiempo_estimado_hrs",
    "peso_kg",
    "volumen_m3",
    "capacidad_kg",
    "capacidad_m3",
    "cantidad_incidencias",
    "tiene_incidencia",
    "uso_capacidad_kg",
    "velocidad_promedio_km_h"
]

df[posibles_predictoras].head()


,distancia_km,tiempo_estimado_hrs,peso_kg,volumen_m3,capacidad_kg,capacidad_m3,cantidad_incidencias,tiene_incidencia,uso_capacidad_kg,velocidad_promedio_km_h
0,919.3,16.7,12675.8,28.48,10000.0,20.5,0.0,0.0,1.26758,55.047904
1,679.6,18.4,13.6,24.93,1000.0,22.1,0.0,0.0,0.01360,36.934783
2,468.4,5.2,2536.0,50.55,20000.0,38.9,0.0,0.0,0.12680,90.076923
3,1642.4,18.9,466.0,13.88,1000.0,25.3,1.0,1.0,0.46600,86.899471
4,1556.6,17.9,4928.9,0.44,5000.0,49.1,0.0,0.0,0.98578,86.960894



## 8. Importante: evitar variables incorrectas

No conviene usar como predictoras variables que sean directamente el resultado final o que puedan generar fuga de información.

Por ejemplo, si se quiere predecir `entrega_tardia`, hay que tener cuidado con variables como:

- `fecha_entrega`
- `dias_entrega`

porque podrían contener información que solo se conoce después de que el envío ya terminó.

Esto se conoce como **data leakage** o fuga de información.

La selección final de variables debe hacerse considerando qué información estaría disponible antes de realizar o completar el envío.



## 9. Conclusión del enfoque

Las instrucciones de la evaluación no indican una variable específica que obligatoriamente se deba predecir. Lo que se solicita es implementar modelos supervisados y no supervisados, evaluarlos, compararlos y justificar técnicamente las decisiones.

Por eso, en este proyecto se propone:

### Modelo de clasificación

Predecir:

`entrega_tardia`

porque permite anticipar si un envío podría llegar tarde.

### Modelo de regresión

Predecir:

`dias_entrega`

porque permite estimar el tiempo de entrega.

Ambos enfoques son coherentes con un problema logístico real y permiten cumplir con los requerimientos de modelado de la Evaluación 2.
